# LES Subgrid-Scale Closure for 2D Decaying Turbulence

This notebook demonstrates **learning a subgrid-scale (SGS) closure model**
for 2D decaying turbulence using a pseudospectral solver.

1. **Part A — A priori training:** Train a neural network offline on exact SGS data
2. **Part B — A posteriori calibration:** Calibrate the model using trajectory matching


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from numpy.fft import fft2, ifft2, fftfreq

torch.manual_seed(0)
np.random.seed(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Parameters

The viscosity $\nu = 10^{-3}$ controls the rate of energy dissipation.
With a broadband turbulent initial condition, the fine $128 \times 128$ simulation
develops small-scale structures that the coarse $32 \times 32$ solver cannot
resolve — the difference is the SGS forcing.


In [ ]:
N_f   = 128
N_c   = 32
L     = 2 * np.pi
nu    = 1e-3
T     = 5.0
N_snap = 60
EPOCHS_PRIORI = 300

ratio = N_f // N_c
dt_f  = 0.256 / N_f   # CFL-scaled: ~2e-3 at N_f=128
dt_c  = 0.256 / N_c   # CFL-scaled: ~8e-3 at N_c=32

print(f"Fine grid: {N_f}x{N_f} = {N_f**2:,} points")
print(f"Coarse grid: {N_c}x{N_c} = {N_c**2:,} points")
print(f"Refinement ratio: {ratio}x per dimension ({ratio**2}x total)")
print(f"Viscosity: {nu}")
print(f"Simulation time: T={T}")
print(f"Timesteps: dt_f={dt_f:.2e}, dt_c={dt_c:.2e}")

## 2. Spectral Utilities

The pseudospectral method computes spatial derivatives exactly in Fourier space.
For a doubly periodic domain, the vorticity-streamfunction formulation is natural:

$$\hat{\psi} = -\hat{\omega} / |\mathbf{k}|^2, \qquad
u = \frac{\partial \psi}{\partial y}, \quad
v = -\frac{\partial \psi}{\partial x}$$

The spectral filter for coarsening keeps only the $N_c/2$ lowest wavenumbers in
each direction — a sharp cutoff in Fourier space.

In [ ]:
def make_wavenumbers(N):
    k = fftfreq(N, d=1.0/N)
    kx, ky = np.meshgrid(k, k, indexing='ij')
    k2 = kx**2 + ky**2
    k2[0, 0] = 1.0   # avoid division by zero (mean mode)
    return kx, ky, k2

KX_f, KY_f, K2_f = make_wavenumbers(N_f)
KX_c, KY_c, K2_c = make_wavenumbers(N_c)

def laplacian_inv(omega_hat, K2):
    """psi_hat = -omega_hat / K2  (streamfunction from vorticity)."""
    psi_hat = -omega_hat / K2
    psi_hat[0, 0] = 0.0
    return psi_hat

def velocity_from_vorticity(omega_hat, KX, KY, K2):
    psi_hat = laplacian_inv(omega_hat, K2)
    u = np.real(ifft2(1j * KY * psi_hat))
    v = np.real(ifft2(-1j * KX * psi_hat))
    return u, v

def jacobian(omega, psi_hat, KX, KY):
    """J(psi, omega) = dpsi/dx * domega/dy - dpsi/dy * domega/dx."""
    omega_hat = fft2(omega)
    dpsi_dx   = np.real(ifft2(1j * KX * psi_hat))
    dpsi_dy   = np.real(ifft2(1j * KY * psi_hat))
    domega_dx = np.real(ifft2(1j * KX * omega_hat))
    domega_dy = np.real(ifft2(1j * KY * omega_hat))
    return dpsi_dx * domega_dy - dpsi_dy * domega_dx

def spectral_filter(omega, N_c, ratio):
    """Low-pass filter: keep only the N_c lowest modes."""
    omega_hat = fft2(omega)
    N_f = omega.shape[0]
    kc  = N_c // 2
    filtered = np.zeros((N_c, N_c), dtype=complex)
    filtered[:kc, :kc]   = omega_hat[:kc, :kc]
    filtered[:kc, -kc:]  = omega_hat[:kc, -kc:]
    filtered[-kc:, :kc]  = omega_hat[-kc:, :kc]
    filtered[-kc:, -kc:] = omega_hat[-kc:, -kc:]
    scale = (N_c / N_f)**2
    return np.real(ifft2(filtered)) * scale

def vorticity_rhs(omega, KX, KY, K2, nu, tau_sgs=None):
    omega_hat = fft2(omega)
    psi_hat   = laplacian_inv(omega_hat, K2)
    J         = jacobian(omega, psi_hat, KX, KY)
    diff      = np.real(ifft2(-nu * K2 * omega_hat))
    rhs       = -J + diff
    if tau_sgs is not None:
        rhs += tau_sgs
    return rhs

def rk4_step(omega, rhs_fn, dt, **kwargs):
    k1 = rhs_fn(omega,          **kwargs)
    k2 = rhs_fn(omega + dt/2*k1, **kwargs)
    k3 = rhs_fn(omega + dt/2*k2, **kwargs)
    k4 = rhs_fn(omega + dt*k3,   **kwargs)
    return omega + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

## 3. Initial Condition: Decaying Turbulence

We initialise the vorticity with a random field whose energy spectrum
follows $E(k) \sim k^{-3}$ (characteristic of the 2D enstrophy cascade).
This places energy at all resolved scales, creating a non-trivial SGS
closure problem when the field is projected onto the coarse grid.


In [ ]:
def turbulent_ic(N, L, seed=42):
    """Random vorticity field with E(k) ~ k^{-3} spectrum."""
    rng = np.random.default_rng(seed)
    k = fftfreq(N, d=1.0 / N)
    kx, ky = np.meshgrid(k, k, indexing='ij')
    k_mag = np.sqrt(kx**2 + ky**2)
    k_mag[0, 0] = 1.0

    amplitude = k_mag**(-2)
    amplitude[0, 0] = 0.0
    amplitude[k_mag > N / 3] = 0.0

    phases = rng.uniform(0, 2 * np.pi, (N, N))
    omega_hat = amplitude * np.exp(1j * phases)
    omega_hat = 0.5 * (omega_hat + np.conj(omega_hat[::-1, ::-1]))

    omega = np.real(ifft2(omega_hat))
    omega *= 5.0 / np.max(np.abs(omega))
    return omega

omega_ic_demo = turbulent_ic(64, L)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(omega_ic_demo.T, origin='lower', cmap='RdBu_r',
               extent=[0, 2*np.pi, 0, 2*np.pi])
plt.colorbar(im, ax=ax)
ax.set_title('Example turbulent initial condition (64x64)')
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()


## 4. Fine-Grid Simulation (DNS)

Run the fine simulation and collect snapshots at regular intervals.


In [ ]:
print("Running fine-grid simulation ...")
omega_f = turbulent_ic(N_f, L, seed=42)
omega_f_ic = omega_f.copy()

snapshots_fine = [omega_f.copy()]
t_snap = np.linspace(0, T, N_snap + 1)[1:]

t = 0.0
snap_idx = 0
while t < T - 1e-10:
    dt = min(dt_f, T - t)
    omega_f = rk4_step(omega_f, vorticity_rhs,
                       dt, KX=KX_f, KY=KY_f, K2=K2_f, nu=nu)
    t += dt
    if snap_idx < N_snap and t >= t_snap[snap_idx] - 1e-10:
        snapshots_fine.append(omega_f.copy())
        snap_idx += 1
        if snap_idx % 10 == 0:
            print(f"  t={t:.2f}")

print(f"Done. Collected {len(snapshots_fine)} snapshots.")


Let's visualise the evolution of the turbulent field — vortex
merging and the inverse cascade transfer energy to larger scales.


In [ ]:
snap_indices = [0, len(snapshots_fine)//4, len(snapshots_fine)//2,
                3*len(snapshots_fine)//4, -1]
snap_times = [0] + [t_snap[i-1] if i > 0 else 0 for i in snap_indices[1:]]

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, idx in zip(axes, snap_indices):
    snap = snapshots_fine[idx]
    t_label = 0.0 if idx == 0 else t_snap[idx - 1] if idx > 0 else T
    im = ax.imshow(snap.T, origin='lower', cmap='RdBu_r',
                   extent=[0, 2*np.pi, 0, 2*np.pi], vmin=-3, vmax=3)
    ax.set_title(f't = {t_label:.1f}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Fine-grid vorticity evolution (Kelvin-Helmholtz instability)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Computing the Exact SGS Stress

The **exact** SGS forcing is the difference between the filtered nonlinear term
and the nonlinear term computed from filtered variables:

$$\tau_{\text{sgs}} = \overline{J(\psi, \omega)} - J(\bar{\psi}, \bar{\omega})$$

This is the "closure term" — it represents the effect of unresolved scales on the
resolved flow.  We compute it from the fine-grid snapshots to create training data.

In [ ]:
def compute_sgs(omega_fine, ratio):
    """Compute exact SGS forcing on coarse grid."""
    N_f = omega_fine.shape[0]
    N_c = N_f // ratio
    KXc, KYc, K2c = make_wavenumbers(N_c)

    omega_bar = spectral_filter(omega_fine, N_c, ratio)

    omega_hat_f = fft2(omega_fine)
    KXf, KYf, K2f = make_wavenumbers(N_f)
    psi_hat_f   = laplacian_inv(omega_hat_f, K2f)
    J_fine      = jacobian(omega_fine, psi_hat_f, KXf, KYf)
    J_fine_bar  = spectral_filter(J_fine, N_c, ratio)

    omega_bar_hat = fft2(omega_bar)
    psi_bar_hat   = laplacian_inv(omega_bar_hat, K2c)
    J_coarse      = jacobian(omega_bar, psi_bar_hat, KXc, KYc)

    # SGS correction: -J_coarse + tau = -J_fine_bar  =>  tau = J_coarse - J_fine_bar
    tau_sgs = J_coarse - J_fine_bar
    return omega_bar, tau_sgs


In [ ]:
print("Computing SGS stress from snapshots ...")
X_data, Y_data = [], []
for snap in snapshots_fine[1:]:
    omega_bar, tau_sgs = compute_sgs(snap, ratio)

    KXc, KYc, K2c = make_wavenumbers(N_c)
    omega_hat = fft2(omega_bar)
    domega_dx = np.real(ifft2(1j * KXc * omega_hat))
    domega_dy = np.real(ifft2(1j * KYc * omega_hat))
    lap_omega = np.real(ifft2(-K2c * omega_hat))

    feat = np.stack([omega_bar, domega_dx, domega_dy, lap_omega], axis=-1)
    X_data.append(feat.reshape(-1, 4))
    Y_data.append(tau_sgs.reshape(-1, 1))

X_data = np.concatenate(X_data, axis=0).astype(np.float32)
Y_data = np.concatenate(Y_data, axis=0).astype(np.float32)

N_train = int(0.85 * len(X_data))
X_tr = torch.tensor(X_data[:N_train]).to(DEVICE)
Y_tr = torch.tensor(Y_data[:N_train]).to(DEVICE)
X_te = torch.tensor(X_data[N_train:]).to(DEVICE)
Y_te = torch.tensor(Y_data[N_train:]).to(DEVICE)

print(f"Training samples: {X_tr.shape[0]:,}")
print(f"Test samples:     {X_te.shape[0]:,}")

Let's visualise the exact SGS forcing for the final snapshot to see what the
network needs to learn:

In [ ]:
omega_bar_demo, tau_demo = compute_sgs(snapshots_fine[-1], ratio)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

im1 = axes[0].imshow(omega_bar_demo.T, origin='lower', cmap='RdBu_r',
                      extent=[0, 2*np.pi, 0, 2*np.pi], vmin=-3, vmax=3)
axes[0].set_title('Filtered vorticity $\\bar{\\omega}$')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(tau_demo.T, origin='lower', cmap='RdBu_r',
                      extent=[0, 2*np.pi, 0, 2*np.pi])
axes[1].set_title('Exact SGS forcing $\\tau_{\\mathrm{sgs}}$')
plt.colorbar(im2, ax=axes[1])

for ax in axes:
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

print(f"SGS forcing magnitude: max={np.abs(tau_demo).max():.3f}, "
      f"std={np.std(tau_demo):.3f}")
print("The SGS forcing is strongest near the vortex cores and braids.")

## 6. SGS Network

The network maps local vorticity features to the SGS forcing:

| Feature | Why |
|---------|-----|
| $\bar{\omega}$ | Local vorticity magnitude |
| $\partial\bar{\omega}/\partial x$ | Streamwise gradient |
| $\partial\bar{\omega}/\partial y$ | Cross-stream gradient |
| $\nabla^2\bar{\omega}$ | Vorticity diffusion / curvature |

Architecture: 4 → 64 → 64 → 32 → 1 with Tanh activations.

In [ ]:
class SGSNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),  nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 32), nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model     = SGSNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS_PRIORI)
loss_fn   = nn.MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(model)

## 7. A Priori Training

In [ ]:
print("A priori training ...")
train_losses_p, test_losses_p = [], []

for epoch in range(EPOCHS_PRIORI):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_tr), Y_tr)
    loss.backward()
    optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        tl = loss_fn(model(X_te), Y_te).item()
    train_losses_p.append(loss.item())
    test_losses_p.append(tl)

    if (epoch+1) % 50 == 0:
        print(f"  Epoch {epoch+1}  train={loss.item():.2e}  test={tl:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(train_losses_p, label='Train', alpha=0.7)
ax.semilogy(test_losses_p,  label='Test',  alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('A priori training convergence')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. A Priori Evaluation

Let's compare the predicted SGS forcing against the exact one for the final snapshot,
then run the coarse simulation with and without the SGS model.

In [ ]:
model.eval()

def predict_sgs(omega_bar, model):
    KXc, KYc, K2c = make_wavenumbers(N_c)
    omega_hat = fft2(omega_bar)
    domega_dx = np.real(ifft2(1j * KXc * omega_hat))
    domega_dy = np.real(ifft2(1j * KYc * omega_hat))
    lap_omega = np.real(ifft2(-K2c * omega_hat))
    feat = np.stack([omega_bar, domega_dx, domega_dy, lap_omega], axis=-1)
    feat_t = torch.tensor(feat.reshape(-1, 4).astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        tau_pred = model(feat_t).cpu().numpy().reshape(N_c, N_c)
    return tau_pred

# A priori comparison on last snapshot
omega_snap = snapshots_fine[-1]
omega_bar_test, tau_exact = compute_sgs(omega_snap, ratio)
tau_pred = predict_sgs(omega_bar_test, model)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

vmax_tau = max(np.abs(tau_exact).max(), np.abs(tau_pred).max())

im1 = axes[0].imshow(tau_exact.T, origin='lower', cmap='RdBu_r',
                      extent=[0, 2*np.pi, 0, 2*np.pi],
                      vmin=-vmax_tau, vmax=vmax_tau)
axes[0].set_title('Exact SGS forcing')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(tau_pred.T, origin='lower', cmap='RdBu_r',
                      extent=[0, 2*np.pi, 0, 2*np.pi],
                      vmin=-vmax_tau, vmax=vmax_tau)
axes[1].set_title('NN predicted SGS forcing')
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow((tau_exact - tau_pred).T, origin='lower', cmap='RdBu_r',
                      extent=[0, 2*np.pi, 0, 2*np.pi])
axes[2].set_title('Prediction error')
plt.colorbar(im3, ax=axes[2])

for ax in axes:
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('A priori SGS prediction quality', fontsize=12)
plt.tight_layout()
plt.show()

corr = np.corrcoef(tau_exact.flatten(), tau_pred.flatten())[0, 1]
print(f"Correlation coefficient: {corr:.3f}")

## 9. Coarse Simulations: No SGS vs A Priori NN

Now we run two coarse simulations from the **same filtered initial condition**:
one with no SGS closure and one with the a priori NN.  Using a consistent
initial condition (filtered DNS) ensures any differences are due to the
SGS closure, not initial condition mismatch.


In [ ]:
print("Running coarse simulations ...")
KXc, KYc, K2c = make_wavenumbers(N_c)

# Use filtered fine IC for consistency (same resolved modes)
omega_c_ic = spectral_filter(omega_f_ic, N_c, ratio)
omega_c_no_sgs = omega_c_ic.copy()
omega_c_sgs    = omega_c_ic.copy()

t = 0.0
while t < T - 1e-10:
    dt = min(dt_c, T - t)

    omega_c_no_sgs = rk4_step(omega_c_no_sgs, vorticity_rhs,
                               dt, KX=KXc, KY=KYc, K2=K2c, nu=nu)

    tau = predict_sgs(omega_c_sgs, model)
    omega_c_sgs = rk4_step(omega_c_sgs, vorticity_rhs,
                            dt, KX=KXc, KY=KYc, K2=K2c, nu=nu, tau_sgs=tau)
    t += dt

omega_ref = spectral_filter(snapshots_fine[-1], N_c, ratio)
print("Done.")


### Energy and Enstrophy Spectra

Two related spectral quantities describe the distribution of turbulent activity across scales.

**Energy spectrum** $E(k)$: kinetic energy per unit wavenumber,
$$E(k) = \frac{1}{2}\sum_{|\mathbf{k}'|\approx k}|\hat{\mathbf{u}}_{\mathbf{k}'}|^2.$$
In 2D turbulence two inertial ranges are predicted by Kraichnan–Leith–Batchelor theory:
- **Enstrophy cascade** (small scales, $k > k_{\rm inj}$): $E(k)\sim k^{-3}$
- **Inverse energy cascade** (large scales, $k < k_{\rm inj}$): $E(k)\sim k^{-5/3}$

**Enstrophy spectrum** $Z(k)$: enstrophy ($= \frac{1}{2}\int|\omega|^2\,dA$) per unit wavenumber,
$$Z(k) = k^2 E(k).$$
Because vorticity and velocity are related in spectral space by $\hat{\omega}_{\mathbf{k}} = i\mathbf{k}\times\hat{\mathbf{u}}_{\mathbf{k}}$
(so $|\hat{\omega}|^2 = k^2|\hat{\mathbf{u}}|^2$), the enstrophy spectrum is simply
$Z(k)=\sum_{|\mathbf{k}'|\approx k}|\hat{\omega}_{\mathbf{k}'}|^2/N^4$.

**What we plot.** We compute $E(k)$ by accumulating $|\hat{\omega}|^2/k^2$ into wavenumber bins
(dividing by $k^2$ converts the naturally available enstrophy spectrum into the energy spectrum).
A well-resolved DNS should show the $k^{-3}$ slope; coarse-grid runs deviate at high $k$
precisely because the SGS closure is missing.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

vmin, vmax = -3, 3
titles = ['Fine (reference, coarsened)', 'Coarse (no SGS)', 'Coarse + NN SGS']
fields = [omega_ref, omega_c_no_sgs, omega_c_sgs]

for col, (title, field) in enumerate(zip(titles, fields)):
    im = axes[0, col].imshow(field.T, origin='lower', cmap='RdBu_r',
                              vmin=vmin, vmax=vmax,
                              extent=[0, 2*np.pi, 0, 2*np.pi])
    axes[0, col].set_title(title, fontsize=11)
    plt.colorbar(im, ax=axes[0, col])

# SGS stress comparison
im1 = axes[1, 0].imshow(tau_exact.T, origin='lower', cmap='RdBu_r',
                         extent=[0, 2*np.pi, 0, 2*np.pi])
axes[1, 0].set_title('Exact SGS forcing $\\tau_{\\rm sgs}$')
plt.colorbar(im1, ax=axes[1, 0])

im2 = axes[1, 1].imshow(tau_pred.T, origin='lower', cmap='RdBu_r',
                         extent=[0, 2*np.pi, 0, 2*np.pi])
axes[1, 1].set_title('NN predicted SGS forcing')
plt.colorbar(im2, ax=axes[1, 1])

# Energy spectra
def energy_spectrum_1d(omega, N):
    omega_hat = fft2(omega)
    k = np.arange(1, N//2)
    E = np.zeros(len(k))
    kx_int = fftfreq(N, d=1.0/N).astype(int)
    ky_int = fftfreq(N, d=1.0/N).astype(int)
    for i, ki in enumerate(k):
        mask = (np.sqrt(kx_int[:,None]**2 + ky_int[None,:]**2) > ki-0.5) & \
               (np.sqrt(kx_int[:,None]**2 + ky_int[None,:]**2) <= ki+0.5)
        E[i] = 0.5 * np.sum(np.abs(omega_hat[mask])**2) / N**4 / ki**2
    return k, E

k_ref, E_ref = energy_spectrum_1d(omega_ref, N_c)
k_no, E_no   = energy_spectrum_1d(omega_c_no_sgs, N_c)
k_sg, E_sg   = energy_spectrum_1d(omega_c_sgs, N_c)

ax = axes[1, 2]
ax.loglog(k_ref, E_ref, 'k-', lw=2, label='Fine (reference)')
ax.loglog(k_no,  E_no,  'r--', label='Coarse (no SGS)')
ax.loglog(k_sg,  E_sg,  'b-',  label='Coarse + NN SGS')
ax.set_xlabel('Wavenumber k'); ax.set_ylabel('E(k)')
ax.set_title('Energy spectrum'); ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('2D Shear Layer LES: NN SGS Closure', fontsize=14)
plt.tight_layout()
plt.show()

err_no = np.mean((omega_ref - omega_c_no_sgs)**2)**0.5
err_sg = np.mean((omega_ref - omega_c_sgs)**2)**0.5
print(f"Vorticity RMSE (no SGS):  {err_no:.4e}")
print(f"Vorticity RMSE (NN SGS):  {err_sg:.4e}")

---
# Part B: A Posteriori Calibration

The a priori model was trained on SGS snapshots from the DNS — data that
comes from the *filtered fine* distribution.  At deployment, the coarse solver
drifts into states the model has never seen, and the unscaled a priori
predictions can overshoot or undershoot.

A simple but effective fix: **calibrate a scaling factor** $\alpha$ so that the
deployed correction is $\tau^{\mathrm{sgs}} = \alpha \cdot f_{\theta}(\bar{\omega})$.
We find $\alpha$ by evaluating rollout accuracy over short trajectory windows
from the DNS reference.

> **Note:** Full end-to-end fine-tuning (back-propagating through the solver)
> is the gold standard for a posteriori training, but 2D turbulence is
> chaotic enough that the short-window gradients can be unreliable.  The
> scaling-factor calibration sidesteps this by reducing the problem to a 1-D
> search.


## 11. A Posteriori Calibration

We sweep a scaling factor $\alpha \in [0, 2]$ and evaluate the rollout
RMSE over all reference windows.  The optimal $\alpha$ minimises the
average endpoint error.

- $\alpha = 0$: no correction (baseline)
- $\alpha = 1$: raw a priori model
- $\alpha < 1$: a priori model overshoots (common due to distribution shift)
- $\alpha > 1$: a priori model undershoots


In [ ]:
reference_coarse = np.array([
    spectral_filter(snap, N_c, ratio) for snap in snapshots_fine
], dtype=np.float32)

dt_ref = T / N_snap
K_window = int(round(dt_ref / dt_c))
n_windows = len(reference_coarse) - 1

print(f"Reference snapshots: {len(reference_coarse)}")
print(f"Coarse steps per window: {K_window}  ({K_window*dt_c:.4f}s)")
print(f"Windows available: {n_windows}")


In [ ]:
# Full-trajectory calibration: run complete coarse simulation at each alpha
alphas = np.linspace(0, 2.0, 21)
rmse_per_alpha = []

model.eval()
print("Full-trajectory calibration sweep ...")

for alpha in alphas:
    omega_c = omega_c_ic.copy()
    t = 0.0
    while t < T - 1e-10:
        dt = min(dt_c, T - t)
        tau = alpha * predict_sgs(omega_c, model)
        omega_c = rk4_step(omega_c, vorticity_rhs,
                           dt, KX=KXc, KY=KYc, K2=K2c, nu=nu, tau_sgs=tau)
        t += dt
    rmse_per_alpha.append(np.sqrt(np.mean((omega_c - omega_ref)**2)))

rmse_per_alpha = np.array(rmse_per_alpha)
best_idx = np.argmin(rmse_per_alpha)
alpha_opt = alphas[best_idx]

print(f"Optimal alpha: {alpha_opt:.2f}")
print(f"RMSE at alpha=0 (no SGS):           {rmse_per_alpha[0]:.4e}")
print(f"RMSE at alpha=1 (raw a priori):     {rmse_per_alpha[alphas==1.0][0]:.4e}")
print(f"RMSE at alpha={alpha_opt:.1f} (calibrated): {rmse_per_alpha[best_idx]:.4e}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(alphas, rmse_per_alpha, 'o-', color='C0')
ax.axvline(1.0, color='C1', ls='--', label=r'$\alpha=1$ (a priori)')
ax.axvline(alpha_opt, color='C2', ls='--', label=r'$\alpha={:.1f}$ (optimal)'.format(alpha_opt))
ax.set_xlabel(r'Scaling factor $\alpha$')
ax.set_ylabel('Rollout RMSE')
ax.set_title('A posteriori calibration: optimal scaling')
ax.legend()
plt.tight_layout()
plt.show()

print(f"The optimal alpha={alpha_opt:.2f} < 1 confirms that the a priori model")
print(f"overshoots — a direct consequence of distribution mismatch between")
print(f"training (filtered DNS states) and deployment (coarse-solver states).")


## 12. A Posteriori Evaluation


In [ ]:
print("Running calibrated a posteriori coarse simulation ...")
model.eval()

omega_c_post = omega_c_ic.copy()
t = 0.0
while t < T - 1e-10:
    dt = min(dt_c, T - t)
    tau = alpha_opt * predict_sgs(omega_c_post, model)
    omega_c_post = rk4_step(omega_c_post, vorticity_rhs,
                            dt, KX=KXc, KY=KYc, K2=K2c, nu=nu, tau_sgs=tau)
    t += dt

rmse_no  = np.mean((omega_ref - omega_c_no_sgs)**2)**0.5
rmse_pri = np.mean((omega_ref - omega_c_sgs)**2)**0.5
rmse_pst = np.mean((omega_ref - omega_c_post)**2)**0.5
print(f"No SGS RMSE:         {rmse_no:.4e}")
print(f"A priori RMSE:       {rmse_pri:.4e}")
print(f"Calibrated RMSE:     {rmse_pst:.4e}")
print(f"A priori improvement:    {rmse_no/rmse_pri:.2f}x")
print(f"Calibrated improvement:  {rmse_no/rmse_pst:.2f}x")


## 13. Full Comparison

**Top row:** vorticity fields.  **Bottom row:** energy spectra.
The calibrated model ($\alpha < 1$) improves on both the uncorrected
coarse simulation and the raw a priori model.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

vmin, vmax = -3, 3
titles2 = ['Filtered DNS', 'No SGS', f'A priori (alpha=1)', f'Calibrated (alpha={alpha_opt:.1f})']
fields2 = [omega_ref, omega_c_no_sgs, omega_c_sgs, omega_c_post]

for j, (title, field) in enumerate(zip(titles2, fields2)):
    im = axes[0, j].imshow(field.T, origin='lower', cmap='RdBu_r',
                            extent=[0, 2*np.pi, 0, 2*np.pi],
                            vmin=vmin, vmax=vmax)
    axes[0, j].set_title(title)
    axes[0, j].set_xlabel('x')
    if j == 0:
        axes[0, j].set_ylabel('y')

plt.colorbar(im, ax=axes[0, :].tolist(), shrink=0.6)

def energy_spectrum_1d(omega, N):
    oh = np.fft.fft2(omega)
    k = np.fft.fftfreq(N, d=1.0/N)
    kx, ky = np.meshgrid(k, k, indexing='ij')
    k_mag = np.sqrt(kx**2 + ky**2)
    k_mag[0, 0] = 1.0  # avoid divide-by-zero at DC
    E_2d = np.abs(oh)**2 / N**4 / k_mag**2  # |ω̂|²/k² gives energy from vorticity
    k_bins = np.arange(0.5, N//2 + 1, 1.0)
    E_1d = np.zeros(len(k_bins) - 1)
    for i in range(len(k_bins) - 1):
        mask = (k_mag >= k_bins[i]) & (k_mag < k_bins[i+1])
        E_1d[i] = E_2d[mask].sum()
    k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])
    return k_centers, E_1d

for j, (title, field) in enumerate(zip(titles2, fields2)):
    k_c, E_c = energy_spectrum_1d(field, N_c)
    axes[1, j].loglog(k_c, E_c, 'C0-', lw=1.5)
    if j > 0:
        k_ref, E_ref = energy_spectrum_1d(omega_ref, N_c)
        axes[1, j].loglog(k_ref, E_ref, 'k--', alpha=0.5, label='DNS ref')
        axes[1, j].legend(fontsize=8)
    axes[1, j].set_xlabel('k')
    if j == 0:
        axes[1, j].set_ylabel('E(k)')
    axes[1, j].set_title(f'Energy spectrum: {title}')

plt.tight_layout()
plt.show()


## 14. RMSE Summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['No SGS', 'A priori NN', f'Calibrated (alpha={alpha_opt:.1f})']
rmses  = [rmse_no, rmse_pri, rmse_pst]
colors = ['C3', 'C0', 'C2']

bars = ax.bar(labels, rmses, color=colors, edgecolor='k', linewidth=0.5)
ax.set_ylabel('RMSE vs filtered DNS')
ax.set_title('Coarse-simulation accuracy')

for bar, val in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print(f"A priori improvement:    {rmse_no/rmse_pri:.2f}x over no SGS")
print(f"Calibrated improvement:  {rmse_no/rmse_pst:.2f}x over no SGS")


## Summary

This notebook demonstrates the full pipeline from DNS data to trained
SGS closure model for 2D decaying turbulence:

1. **Fine DNS** provides ground-truth snapshots
2. **Spectral filtering** projects DNS data onto the coarse grid
3. **A priori training** teaches the NN to predict the SGS forcing
   from local coarse-grid features — achieving modest but consistent
   RMSE improvement
4. **A posteriori calibration** reveals that the raw a priori model
   overshoots (optimal $\alpha < 1$), a direct consequence of
   **distribution mismatch** between training and deployment states

**Key takeaway:** even a simple local model (MLP on 4 features) can learn
a useful SGS correction, but the a priori → a posteriori gap matters.
Calibrating the deployment scaling from trajectory data provides a
straightforward path to improved accuracy.
